# Fashion E-commerce Analytics — Data Cleaning

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
BASE_DIR = Path("..")

RAW_DIR = BASE_DIR / "data" / "raw"
CLEANED_DIR = BASE_DIR / "data" / "cleaned"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
customers = pd.read_csv(RAW_DIR / "customers.csv")
products = pd.read_csv(RAW_DIR / "products.csv")
discounts = pd.read_csv(RAW_DIR / "discounts.csv")
employees = pd.read_csv(RAW_DIR / "employees.csv")
stores = pd.read_csv(RAW_DIR / "stores.csv")

transactions = pd.read_csv(
    RAW_DIR / "transactions.csv",
    parse_dates=["Date"]
)

print("Customers :", customers.shape)
print("Products :", products.shape)
print("Discounts :", discounts.shape)
print("Employees :", employees.shape)
print("Stores :", stores.shape)
print("Transactions :", transactions.shape)

C:\Users\dell\AppData\Local\Temp\ipykernel_22996\1843591346.py:1: DtypeWarning: Columns (0: Telephone) have mixed types. Specify dtype option on import or set low_memory=False.
  customers = pd.read_csv(RAW_DIR / "customers.csv")


Customers : (1643306, 9)
Products : (17940, 12)
Discounts : (181, 6)
Employees : (404, 4)
Stores : (35, 8)
Transactions : (6416827, 19)


In [4]:
customers_clean = customers.copy()

customers_clean["Job Title"] = (
    customers_clean["Job Title"]
    .fillna("Unknown")
)

In [5]:
products_clean = products.copy()

products_clean["Color"] = (
    products_clean["Color"]
    .fillna("Unknown")
)

products_clean["Sizes"] = (
    products_clean["Sizes"]
    .fillna("N/A")
)

In [6]:
discounts_clean = discounts.copy()

discounts_clean["Category"] = (
    discounts_clean["Category"]
    .fillna("All")
)

discounts_clean["Sub Category"] = (
    discounts_clean["Sub Category"]
    .fillna("All")
)

In [7]:
transactions_clean = transactions.copy()

transactions_clean["Color"] = (
    transactions_clean["Color"]
    .fillna("Unknown")
)

transactions_clean["Size"] = (
    transactions_clean["Size"]
    .fillna("Unknown")
)

In [8]:
datasets_clean = {
    "customers": customers_clean,
    "products": products_clean,
    "discounts": discounts_clean,
    "employees": employees,
    "stores": stores,
    "transactions": transactions_clean
}

for name, df in datasets_clean.items():
    print(f"\n===== {name.upper()} =====")
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print(missing if len(missing) > 0 else "Aucune valeur manquante")


===== CUSTOMERS =====
Aucune valeur manquante

===== PRODUCTS =====
Aucune valeur manquante

===== DISCOUNTS =====
Aucune valeur manquante

===== EMPLOYEES =====
Aucune valeur manquante

===== STORES =====
Aucune valeur manquante

===== TRANSACTIONS =====
Aucune valeur manquante


In [9]:
customers_clean.to_csv(
    CLEANED_DIR / "customers_clean.csv",
    index=False
)

products_clean.to_csv(
    CLEANED_DIR / "products_clean.csv",
    index=False
)

discounts_clean.to_csv(
    CLEANED_DIR / "discounts_clean.csv",
    index=False
)

employees.to_csv(
    CLEANED_DIR / "employees_clean.csv",
    index=False
)

stores.to_csv(
    CLEANED_DIR / "stores_clean.csv",
    index=False
)

transactions_clean.to_csv(
    CLEANED_DIR / "transactions_clean.csv",
    index=False
)

In [10]:
for file in CLEANED_DIR.glob("*.csv"):
    print(file.name, "→", round(file.stat().st_size / (1024**2), 2), "MB")

customers_clean.csv → 187.19 MB
discounts_clean.csv → 0.02 MB
employees_clean.csv → 0.01 MB
products_clean.csv → 4.86 MB
stores_clean.csv → 0.0 MB
transactions_clean.csv → 805.78 MB


In [11]:
print("===== VALIDATION DES LIGNES =====")

print("Customers :", len(customers), "→", len(customers_clean))
print("Products :", len(products), "→", len(products_clean))
print("Discounts :", len(discounts), "→", len(discounts_clean))
print("Employees :", len(employees), "→", len(employees))
print("Stores :", len(stores), "→", len(stores))
print("Transactions :", len(transactions), "→", len(transactions_clean))

===== VALIDATION DES LIGNES =====
Customers : 1643306 → 1643306
Products : 17940 → 17940
Discounts : 181 → 181
Employees : 404 → 404
Stores : 35 → 35
Transactions : 6416827 → 6416827


In [12]:
print("===== VALIDATION DES VALEURS MANQUANTES =====")

for name, df in datasets_clean.items():
    total_missing = df.isna().sum().sum()
    print(f"{name}: {total_missing} valeurs manquantes")

===== VALIDATION DES VALEURS MANQUANTES =====
customers: 0 valeurs manquantes
products: 0 valeurs manquantes
discounts: 0 valeurs manquantes
employees: 0 valeurs manquantes
stores: 0 valeurs manquantes
transactions: 0 valeurs manquantes


In [13]:
print("Avant suppression :", len(transactions_clean))

duplicate_count = transactions_clean.duplicated().sum()

print("Doublons exacts :", duplicate_count)

Avant suppression : 6416827
Doublons exacts : 798


In [14]:
transactions_clean = transactions_clean.drop_duplicates().reset_index(drop=True)

print("Après suppression :", len(transactions_clean))
print("Doublons restants :", transactions_clean.duplicated().sum())

Après suppression : 6416029
Doublons restants : 0


In [15]:
print("===== TRANSACTIONS PAR TYPE =====")

print(
    transactions_clean["Transaction Type"]
    .value_counts()
)

print("\n===== TOTAL PAR TYPE =====")

print(
    transactions_clean
    .groupby("Transaction Type")["Line Total"]
    .agg(["count", "sum", "mean", "min", "max"])
)

===== TRANSACTIONS PAR TYPE =====
Transaction Type
Sale      6077200
Return     338829
Name: count, dtype: int64

===== TOTAL PAR TYPE =====
                    count           sum        mean     min     max
Transaction Type                                                   
Return             338829 -4.321212e+07 -127.533698 -3348.0    -1.4
Sale              6077200  7.760605e+08  127.700343     1.4  3460.5


In [16]:
print("\n===== COHERENCE DES VENTES =====")

sales = transactions_clean[
    transactions_clean["Transaction Type"] == "Sale"
].copy()

sales["Expected Line Total"] = (
    sales["Unit Price"]
    * sales["Quantity"]
    * (1 - sales["Discount"])
)

sales["Difference"] = (
    sales["Line Total"]
    - sales["Expected Line Total"]
)

print(
    "Incohérences Sale :",
    (sales["Difference"].abs() > 0.01).sum()
)


===== COHERENCE DES VENTES =====
Incohérences Sale : 0


In [17]:
returns = transactions_clean[
    transactions_clean["Transaction Type"] == "Return"
].copy()

returns["Expected Return Total"] = (
    -returns["Unit Price"]
    * returns["Quantity"]
    * (1 - returns["Discount"])
)

returns["Difference"] = (
    returns["Line Total"]
    - returns["Expected Return Total"]
)

print("Nombre de retours :", len(returns))

print(
    "Retours incohérents :",
    (returns["Difference"].abs() > 0.01).sum()
)

print("\nDistribution des différences :")
print(
    returns["Difference"]
    .round(2)
    .value_counts()
    .head(20)
)

Nombre de retours : 338829
Retours incohérents : 97931

Distribution des différences :
Difference
0.00     240898
18.00       595
19.25       536
10.50       527
17.50       504
12.00       502
21.00       499
15.00       496
15.75       476
20.25       467
17.00       464
20.00       452
11.25       449
11.00       440
14.00       432
16.25       431
16.00       429
18.25       422
18.50       421
18.75       416
Name: count, dtype: int64


In [18]:
print(
    returns[
        returns["Difference"].abs() > 0.01
    ][
        [
            "Invoice ID",
            "Unit Price",
            "Quantity",
            "Discount",
            "Line Total",
            "Expected Return Total",
            "Difference"
        ]
    ].head(20)
)

              Invoice ID  Unit Price  Quantity  Discount  Line Total  \
15   RET-US-001-03558764        75.0         1       0.0       -45.0   
16   RET-US-001-03558764        45.5         1       0.0       -27.3   
40   RET-US-001-03558767        54.0         1       0.0       -32.4   
103  RET-US-001-03558799        42.0         1       0.0       -25.2   
109  RET-US-001-03558783        37.0         1       0.0       -22.2   
110  RET-US-001-03558783        61.0         1       0.0       -36.6   
155  RET-US-001-03558769        34.0         1       0.0       -20.4   
208  RET-US-001-03558814        46.0         1       0.0       -27.6   
209  RET-US-001-03558814        53.5         1       0.0       -32.1   
319  RET-US-001-03558813        30.5         1       0.0       -18.3   
383  RET-US-001-03559002        50.5         3       0.0       -90.9   
384  RET-US-001-03559002        40.5         1       0.0       -24.3   
410  RET-US-001-03558978        35.5         1       0.0       -

In [19]:
returns["Return Ratio"] = (
    returns["Line Total"].abs()
    / (returns["Unit Price"] * returns["Quantity"])
)

print(
    returns["Return Ratio"]
    .round(3)
    .value_counts()
    .head(20)
)

Return Ratio
1.000    240898
0.500     48884
0.550     12803
0.650     11879
0.800      7809
0.750      7490
0.600      6770
0.400      2240
0.749        43
0.751        13
Name: count, dtype: int64


In [20]:
print(
    returns["Return Ratio"]
    .describe()
)

count    338829.000000
mean          0.876455
std           0.201594
min           0.400000
25%           0.750000
50%           1.000000
75%           1.000000
max           1.000000
Name: Return Ratio, dtype: float64


In [21]:
print(
    returns[
        returns["Difference"].abs() > 0.01
    ][
        [
            "Unit Price",
            "Quantity",
            "Discount",
            "Line Total",
            "Return Ratio"
        ]
    ].head(30)
)

     Unit Price  Quantity  Discount  Line Total  Return Ratio
15         75.0         1       0.0       -45.0           0.6
16         45.5         1       0.0       -27.3           0.6
40         54.0         1       0.0       -32.4           0.6
103        42.0         1       0.0       -25.2           0.6
109        37.0         1       0.0       -22.2           0.6
110        61.0         1       0.0       -36.6           0.6
155        34.0         1       0.0       -20.4           0.6
208        46.0         1       0.0       -27.6           0.6
209        53.5         1       0.0       -32.1           0.6
319        30.5         1       0.0       -18.3           0.6
383        50.5         3       0.0       -90.9           0.6
384        40.5         1       0.0       -24.3           0.6
410        35.5         1       0.0       -21.3           0.6
411        33.0         1       0.0       -19.8           0.6
412        49.0         1       0.0       -29.4           0.6
470     

In [22]:
transactions_clean["Return Ratio"] = 0.0

return_mask = transactions_clean["Transaction Type"] == "Return"

transactions_clean.loc[return_mask, "Return Ratio"] = (
    transactions_clean.loc[return_mask, "Line Total"].abs()
    / (
        transactions_clean.loc[return_mask, "Unit Price"]
        * transactions_clean.loc[return_mask, "Quantity"]
    )
)

In [23]:
print(
    transactions_clean
    .groupby("Transaction Type")["Return Ratio"]
    .describe()
)

                      count      mean       std  min   25%  50%  75%  max
Transaction Type                                                         
Return             338829.0  0.876455  0.201594  0.4  0.75  1.0  1.0  1.0
Sale              6077200.0  0.000000  0.000000  0.0  0.00  0.0  0.0  0.0


In [25]:
print("===== CONTRÔLE FINAL TRANSACTIONS =====")

print("Nombre de lignes :", len(transactions_clean))

print(
    "Valeurs manquantes :",
    transactions_clean.isna().sum().sum()
)

print(
    "Doublons exacts :",
    transactions_clean.duplicated().sum()
)

print(
    "Dates invalides :",
    transactions_clean["Date"].isna().sum()
)

print(
    "Types de transactions :"
)

print(
    transactions_clean["Transaction Type"].value_counts()
)
transactions_clean.to_csv(
    CLEANED_DIR / "transactions_clean.csv",
    index=False
)

===== CONTRÔLE FINAL TRANSACTIONS =====
Nombre de lignes : 6416029
Valeurs manquantes : 0
Doublons exacts : 0
Dates invalides : 0
Types de transactions :
Transaction Type
Sale      6077200
Return     338829
Name: count, dtype: int64
